## Make Charts
* To run a function in a Makefile, go do make rt_sched_pipeline

In [24]:
import dask.dataframe as dd
import pandas as pd
import yaml
from segment_speed_utils import gtfs_schedule_wrangling, helpers
from segment_speed_utils.project_vars import RT_SCHED_GCS, SCHED_GCS, SEGMENT_GCS
from shared_utils import portfolio_utils, rt_dates, rt_utils

In [25]:
# Times
import datetime

from loguru import logger

In [ ]:
import altair as alt
from calitp_data_analysis import calitp_color_palette

### Trips

In [26]:
# Read in the files
dec_trips = pd.read_parquet(
    "gs://calitp-analytics-data/data-analyses/rt_vs_schedule/vp_trip/trip_metrics/trip_2023-12-13.parquet"
)

In [27]:
dec_trips.sample()

,schedule_gtfs_dataset_key,trip_instance_key,rt_service_min,min_w_atleast2_trip_updates,total_pings,total_min_w_gtfs,total_vp,vp_in_shape,route_id,direction_id,sched_rt_category,sched_time_of_day,service_minutes,rt_time_of_day,time_of_day,pings_per_min,spatial_accuracy_pct,rt_w_gtfs_pct,rt_v_scheduled_time_pct
34260,1770249a5a2e770ca90628434d4934b1,fc7812f09c182e1af19b8f28927f605a,32,31,94,32,94,94,3394,1,vp_sched,Midday,28.0,Midday,Midday,2.9375,1.0,1.0,0.142857


In [28]:
dec_trips.sched_rt_category.value_counts()

vp_sched    77977
vp_only      8151
Name: sched_rt_category, dtype: int64

In [29]:
jan_trips = pd.read_parquet(
    "gs://calitp-analytics-data/data-analyses/rt_vs_schedule/vp_trip/trip_metrics/trip_2024-01-17.parquet"
)

In [30]:
jan_trips.sample()

,schedule_gtfs_dataset_key,trip_instance_key,rt_service_min,min_w_atleast2_trip_updates,total_pings,total_min_w_gtfs,total_vp,vp_in_shape,route_id,direction_id,sched_rt_category,service_minutes,time_of_day,peak_offpeak,pings_per_min,spatial_accuracy_pct,rt_w_gtfs_pct,rt_v_scheduled_time_pct
13940,1c7027faabfeec976ea388973100bcf3,72366297a4238baed48872ee50dbb01c,68,30,72,42,72,71,20cc,0,vp_sched,67.0,PM Peak,peak,1.058824,0.986111,0.617647,0.014925


In [31]:
jan_trips.sched_rt_category.value_counts()

vp_sched    76739
vp_only     10511
Name: sched_rt_category, dtype: int64

In [32]:
feb_trips = pd.read_parquet(
    "gs://calitp-analytics-data/data-analyses/rt_vs_schedule/vp_trip/trip_metrics/trip_2024-02-14.parquet"
)

In [33]:
feb_trips.sample()

,schedule_gtfs_dataset_key,trip_instance_key,rt_service_min,min_w_atleast2_trip_updates,total_pings,total_min_w_gtfs,total_vp,vp_in_shape,route_id,direction_id,sched_rt_category,service_minutes,time_of_day,peak_offpeak,pings_per_min,spatial_accuracy_pct,rt_w_gtfs_pct,rt_v_scheduled_time_pct
64873,ecb6e412d4745e9ebbfb9df814e336f2,5519eaf070ee98613ed48ec2a31a9c14,23,22,66,22,66,63,354,0,vp_sched,24.0,Midday,offpeak,2.869565,0.954545,0.956522,-0.041667


In [34]:
feb_trips.sched_rt_category.value_counts()

vp_sched    78858
vp_only      7761
Name: sched_rt_category, dtype: int64

### Routes

In [35]:
pct_cols = [
    "rt_w_gtfs_pct",
    "rt_v_scheduled_time_pct",
    "spatial_accuracy_pct",
]

In [36]:
int_cols = ["rt_service_min", "service_minutes", "pings_per_min"]

In [37]:
def clean_df(df: pd.DataFrame, pct_cols: list, int_cols: list) -> pd.DataFrame:
    for i in pct_cols:
        df[i] = (df[i] * 100).round(0)
    for i in int_cols:
        df[i] = df[i].fillna(0).round(0).astype(int)

    df.columns = df.columns.str.replace("_", " ").str.strip().str.title()
    return df

In [38]:
feb_routes = pd.read_parquet(
    "gs://calitp-analytics-data/data-analyses/rt_vs_schedule/vp_route_dir/route_direction_metrics/route_2024-02-14.parquet"
)

In [39]:
feb_routes.sample()

,time_period,schedule_gtfs_dataset_key,route_id,direction_id,total_min_w_gtfs,rt_service_min,total_pings,service_minutes,total_vp,vp_in_shape,n_trips,pings_per_min,spatial_accuracy_pct,rt_w_gtfs_pct,rt_v_scheduled_time_pct
758,all_day,4c6b107352b318297bb39173c796f357,3786,0,1825,1930,5398,1734.0,5398,4921,28,2.796891,0.911634,0.945596,0.113033


In [40]:
feb_routes2 = clean_df(feb_routes, pct_cols, int_cols)

In [67]:
def summarize(df: pd.DataFrame, groupby_col: str, agg_col:str, new_col_name:str) -> pd.DataFrame:
    df2 = (df.groupby([groupby_col]).agg({agg_col: "count"}).reset_index()).rename(
        columns={agg_col: new_col_name}
    )
    return df2

In [81]:
# Basic bar chart: input df, x column, y column, column for color code, and chart title.
def basic_bar_chart(df, x_col, y_col, color_col):
    chart_title = f"{x_col} by {y_col}"
    chart = (
        alt.Chart(df)
        .mark_bar()
        .encode(
            x=alt.X(f"{x_col}:O", title=(x_col), sort=("x")),
            y=alt.Y(y_col, title=(y_col)),
            color=alt.Color(
                color_col,
                scale=alt.Scale(range=calitp_color_palette.CALITP_SEQUENTIAL_COLORS),
                legend=alt.Legend(title=((color_col))),
            ),
        )
        .properties(title=chart_title, width=800, height=600)
    )

    return chart

In [82]:
def aggregate_chart(df: pd.DataFrame, groupby_col: str, agg_col:str, new_col_name:str):
    df2 = summarize(df, groupby_col, agg_col, new_col_name)
    chart = basic_bar_chart(df2, groupby_col, new_col_name,new_col_name)
    display(chart)

In [83]:
feb_routes2.sample()

,Time Period,Schedule Gtfs Dataset Key,Route Id,Direction Id,Total Min W Gtfs,Rt Service Min,Total Pings,Service Minutes,Total Vp,Vp In Shape,N Trips,Pings Per Min,Spatial Accuracy Pct,Rt W Gtfs Pct,Rt V Scheduled Time Pct
1771,all_day,baeeb157e85a901e47b828ef9fe75091,84,0,398,403,858,261,858,843,12,2,98.0,99.0,54.0


In [84]:
aggregate_chart(feb_routes2, "Rt W Gtfs Pct", "Route Id", "Number of Routes")

alt.Chart(...)

In [85]:
jan_routes = pd.read_parquet(
    "gs://calitp-analytics-data/data-analyses/rt_vs_schedule/vp_route_dir/route_direction_metrics/route_2024-01-17.parquet"
)

In [86]:
jan_routes.sample()

,time_period,schedule_gtfs_dataset_key,route_id,direction_id,total_min_w_gtfs,rt_service_min,total_pings,service_minutes,total_vp,vp_in_shape,n_trips,pings_per_min,spatial_accuracy_pct,rt_w_gtfs_pct,rt_v_scheduled_time_pct
2210,all_day,cc53a0dbf5df90e3009b9cb5d89d80ba,1458,1,433,434,1285,263.0,1285,908,5,2.960829,0.706615,0.997696,0.65019
